In [ ]:
from google.colab import files


import pandas as pd
import unicodedata
import re
import os
from datasets import Dataset


if os.path.exists("Expanded_Intent_Dataset_2.csv"):
  os.remove("Expanded_Intent_Dataset_2.csv")




uploaded = files.upload()


# ✅ 1. Φόρτωση CSV ΜΙΑ φορά
df_raw = pd.read_csv("Expanded_Intent_Dataset_2.csv", quotechar='"')

# ✅ 2. Normalization
def normalize_text(text):
    text = unicodedata.normalize('NFD', str(text))  # ensure string
    text = ''.join([c for c in text if unicodedata.category(c) != 'Mn'])  # remove accents
    return text.lower()

df = df_raw.copy()  # Κράτησε το raw ανέγγιχτο

# ✅ 3. Καθάρισμα text & intent
df["text"] = df["text"].apply(normalize_text)

df["intent"] = (
    df["intent"]
    .apply(normalize_text)
    .astype(str)
    .str.strip()
    .apply(lambda x: re.sub(r"[\t\n\r\s]+", "", x))       # remove whitespaces, tabs, newlines
    .apply(lambda x: re.sub(r"[^\w_α-ωΑ-Ω]", "", x))       # remove special characters except _
)

# ✅ 4. Έλεγχος μοναδικών intents και συχνοτήτων
print("📌 Μοναδικά intents:", df["intent"].nunique())
pd.set_option('display.max_rows', None)


print(df["intent"].value_counts())

# ✅ 5. Convert to HuggingFace Dataset
dataset = Dataset.from_pandas(df)

Saving Expanded_Intent_Dataset_2.csv to Expanded_Intent_Dataset_2.csv
📌 Μοναδικά intents: 102
intent
πιστοποιητικο_γεννησης                                    78
καδοι_καθαριοτητας                                        78
πιστοποιητικο_οικογενειακης_καταστασης                    77
αδεσποτα_ζωα                                              75
επισκευη_οδοστρωματος                                     73
αδεια_μικρης_κλιμακας                                     72
διευθυνση_οικοπεδου                                       72
μητρωο_αρρενων                                            72
παιδικες_χαρες                                            72
ληξιαρχικη_πραξη_θανατου                                  71
δημοτολογιο                                               71
βεβαιωση_οριων_ιδιοκτησιας                                71
τοποθετηση_δημοτικου_φωτισμου                             71
επεκταση_δικτυου_υδρευσης                                 71
εισοδος_εξοδος                               

In [ ]:
!pip install -q datasets

from datasets import Dataset
from sklearn.preprocessing import LabelEncoder

import re

# Συνάρτηση για normalize
def normalize_text(text):
    text = unicodedata.normalize('NFD', text)
    text = ''.join([c for c in text if unicodedata.category(c) != 'Mn'])
    return text.lower()

df["text"] = df["text"].apply(normalize_text)

#df["intent"] = df["intent"].apply(normalize_text)

# Καθάρισε whitespaces και "σκουπίδια"
#df["intent"] = df["intent"].astype(str).str.strip().str.lower()

# Αφαίρεσε παράξενους χαρακτήρες
#df["intent"] = df["intent"].apply(lambda x: re.sub(r"[^a-zA-Z0-9_α-ωΑ-Ω]", "", x))

# Δες μοναδικές τιμές
print(df["intent"].unique())

# Κωδικοποίηση των intents σε αριθμούς
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["intent"])

# ✅ Μετατροπή σε HuggingFace Dataset
dataset = Dataset.from_pandas(df)

# Έλεγχος
dataset

['πιστοποιητικο_οικογενειακης_καταστασης' 'πιστοποιητικο_γεννησης'
 'επισκευη_οδοστρωματος' 'φωτισμος' 'επικοινωνια'
 'ληξιαρχικη_πραξη_θανατου' 'ληξιαρχικη_πραξη_γαμου' 'αδεια_οικοδομης'
 'βεβαιωση_χρησης_γης' 'βεβαιωση_οριων_ιδιοκτησιας'
 'βεβαιωση_ορων_δομησης' 'διακοπη_υδροδοτησης' 'συνδεση_φυσικου_αεριου'
 'τοποθετηση_δημοτικου_φωτισμου' 'επεκταση_δικτυου_υδρευσης'
 'βλαβη_αγωγου_υδρευσης' 'ληξιαρχειο' 'καδοι_καθαριοτητας'
 'αδεια_μικρης_κλιμακας' 'γενικες_υπηρεσιες' 'αδεσποτα_ζωα' 'δημοτολογιο'
 'πλησιεστεροι_συγγενεις' 'βεβαιωση_κατοικιας' 'μητρωο_αρρενων'
 'τεχνικη_υπηρεσια' 'αγωγοι_αποχετευσης' 'διευθυνση_οικοπεδου'
 'εισοδος_εξοδος' 'εκτος_σχεδιου' 'παλαιοτητα_κτισματος' 'παλαιοτητα_1955'
 'παλαιοτητα_1990' 'παλαιοτητα_οδου' 'παλαιοτητα_οικοπεδου' 'υψομετρο'
 'χιλιομετρικη_αποσταση' 'καταληψη_κοινοχρηστου' 'ασφαλτοστρωση_οδου'
 'καθρεφτης_ορατοτητας' 'παγκακια' 'παιδικες_χαρες' 'σημανση_οδοστρωματος'
 'σημανση_ονοματοθεσιας' 'στεγαστρο_οασα' 'πεζοδρομιο'
 'καθαρισμος_φρεατιου

Dataset({
    features: ['text', 'intent', 'label'],
    num_rows: 7039
})

In [ ]:
# Χωρίζουμε σε train και test
dataset = dataset.train_test_split(test_size=0.2)

# Έλεγχος
dataset["train"][0]


{'text': 'μπορει καποιος απο τον δημο να επιθεωρησει το δρομο μας;',
 'intent': 'επισκευη_οδοστρωματος',
 'label': 51}

In [ ]:
!pip install -q transformers

from transformers import AutoTokenizer

# Χρησιμοποιούμε multilingual μοντέλο γιατί έχουμε ελληνικά
model_name = "nlpaueb/bert-base-greek-uncased-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenization function
def tokenize(example):
    return tokenizer(example["text"], truncation=True)

# Εφαρμογή tokenization
tokenized_dataset = dataset.map(tokenize)


config.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/5631 [00:00<?, ? examples/s]

Map:   0%|          | 0/1408 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification

# Πόσα labels έχουμε;
num_labels = len(label_encoder.classes_)

# Φορτώνουμε το DistilBERT με σωστό αριθμό labels
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)


pytorch_model.bin:   0%|          | 0.00/454M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/454M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/bert-base-greek-uncased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were 

In [ ]:
!pip install -q evaluate

import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy.compute(predictions=predictions, references=labels)
    f1_macro = f1.compute(predictions=predictions, references=labels, average='macro')

    return {
        "accuracy": acc["accuracy"],
        "f1_macro": f1_macro["f1"]
    }


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch", # Changed save_strategy to match eval_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=7,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=10,


)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

trainer.train()



Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.826941,1.563512,0.774858,0.764135
2,0.930734,0.800269,0.827415,0.822698
3,0.527815,0.615707,0.852983,0.849021
4,0.350735,0.553392,0.870028,0.866428
5,0.295734,0.535332,0.870739,0.866797
6,0.233197,0.546458,0.871449,0.867626
7,0.144110,0.551010,0.867898,0.864083


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=4928, training_loss=0.825933867521197, metrics={'train_runtime': 487.8304, 'train_samples_per_second': 80.801, 'train_steps_per_second': 10.102, 'total_flos': 373404343225140.0, 'train_loss': 0.825933867521197, 'epoch': 7.0})

In [ ]:
# αφού έχεις fit τον LabelEncoder
id2label = {i: lbl for i, lbl in enumerate(label_encoder.classes_)}
label2id = {lbl: i for i, lbl in id2label.items()}

model.config.id2label = id2label
model.config.label2id = label2id

model.save_pretrained("dimos-intent-model")
tokenizer.save_pretrained("dimos-intent-model")



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('dimos-intent-model/tokenizer_config.json',
 'dimos-intent-model/tokenizer.json')

In [ ]:
from transformers import pipeline
import unicodedata

def normalize_text(s):
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.lower()

clf = pipeline("text-classification", model="dimos-intent-model", tokenizer="dimos-intent-model")

text = "δεν εχω που να πεταξω τα σκουπιδια μου"
out = clf(normalize_text(text))[0]

predicted_label = out["label"]
score = out["score"]

print(out)
print(f'🗣 "{text}"\n👉 Predicted Intent: {predicted_label} (score: {score:.3f})')


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'label': 'καδοι_καθαριοτητας', 'score': 0.9673978090286255}
🗣 "δεν εχω που να πεταξω τα σκουπιδια μου"
👉 Predicted Intent: καδοι_καθαριοτητας (score: 0.967)


In [ ]:
import pandas as pd
import unicodedata
import re
from sklearn.preprocessing import LabelEncoder

# ✅ Φόρτωση αρχείου
df = pd.read_csv("Expanded_Intent_Dataset_2.csv", quotechar='"')

# ✅ Normalization function
def normalize_text(text):
    text = unicodedata.normalize('NFD', str(text))
    text = ''.join([c for c in text if unicodedata.category(c) != 'Mn'])  # remove accents
    return text.lower()

# ✅ Καθαρισμός text & intent
df["text"] = df["text"].apply(normalize_text)

df["intent"] = (
    df["intent"]
    .apply(normalize_text)
    .astype(str)
    .str.strip()
    .apply(lambda x: re.sub(r"[\t\n\r\s]+", "", x))       # remove whitespaces, tabs, newlines
    .apply(lambda x: re.sub(r"[^\w_α-ωΑ-Ω]", "", x))       # remove special characters except _
)

# ✅ Κωδικοποίηση intents σε αριθμούς
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["intent"])

# ✅ Εκτύπωση mapping label -> intent
id2intent = dict(enumerate(label_encoder.classes_))

print("🔢 Mapping από label σε intent:\n")
for label, intent in id2intent.items():
    print(f"{label:2} → {intent}")


🔢 Mapping από label σε intent:

 0 → nan
 1 → αγορα_οικογενειακου_ταφου
 2 → αγωγοι_αποχετευσης
 3 → αδεια_κολυμβητικης_δεξαμενης
 4 → αδεια_μικρης_κλιμακας
 5 → αδεια_οικοδομης
 6 → αδεια_παιδοτοπου
 7 → αδεια_παραγωγου_αγροτη
 8 → αδεια_πωλητη_υπαιθριου_εμποριου
 9 → αδεσποτα_ζωα
10 → αιτηση_γενικου_τεχνικου_περιεχομενου
11 → αιτηση_εκταφης
12 → αιτηση_κιτρινης_διαγραμμισης
13 → αιτηση_παραχωρησης_αποκλειστικης_θεσης_σταθμευσης_αμεα
14 → αιτηση_προσωρινης_χρησης_οδοστρωματος_πεζοδρομιου
15 → αιτηση_ταφης_σε_οικογενειακο_ταφο
16 → αιτηση_χορηγησης_αντιγραφου_ηδη_χορηγηθεισας_βεβαιωσης
17 → αλλαγη_ιδιοκτησιας_ακινητου
18 → αλλαγη_στοιχειων_λογαριασμου_υδρευσης
19 → αναγγελια_κομμωτη
20 → αναγγελια_τεχνιτη_ακρων
21 → αντιγραφα_αδειας_πολεοδομιας
22 → απαλλαγη_δημοτικων_τελων_covid
23 → ασφαλτοστρωση_οδου
24 → βεβαιωση_αρτιοτητας
25 → βεβαιωση_δραστηριοποιησης_υπαιθριου
26 → βεβαιωση_εγκαταστασης_καταστηματος
27 → βεβαιωση_κατοικιας
28 → βεβαιωση_οριων_ιδιοκτησιας
29 → βεβαιωση_ορων_δομη

In [ ]:
!zip -r dimos-intent-model.zip dimos-intent-model/


In [ ]:
from transformers import pipeline
import unicodedata

def normalize_text(s):
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.lower()

clf = pipeline("text-classification",
               model="dimos-intent-model",
               tokenizer="dimos-intent-model")

text = "η γυναικα μου γεννησε τι πρεπει να κανω τωρα"

# ΜΟΝΟ Top-1
top1 = clf(normalize_text(text), top_k=1)[0]
print(f'Top-1 → {top1["label"]} (score: {top1["score"]:.3f})')

# ή Top-5
top5 = clf(normalize_text(text), top_k=5)
for r in top5:
    print(f'{r["label"]}: {r["score"]:.3f}')


Device set to use cuda:0


Top-1 → πιστοποιητικο_γεννησης (score: 0.800)
πιστοποιητικο_γεννησης: 0.800
πιστοποιητικο_οικογενειακης_καταστασης: 0.024
ληξιαρχικη_πραξη_γαμου: 0.015
ληξιαρχειο: 0.009
πλησιεστεροι_συγγενεις: 0.006
